In [4]:
import numpy as np
import math

NT = ['A', 'C', 'G', 'T']
ST = ['E', '5', 'I']

INIT = {'E': 1.0, '5': 0.0, 'I': 0.0}

TRANS = {
    'S': {'E': 1.0, '5': 0, 'I': 0.0, 'End': 0.0},
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0, 'I': 0.9, 'End': 0.1}
}

EMIT = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

def ln(x):
    return -math.inf if x == 0 else math.log(x)

seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
path_given = "EEEEEEEEEEEEEEEEEE5IIIIIII"


lp = 0.0
prev = 'S'
for c, o in zip(path_given, seq):
    lp += ln(TRANS[prev][c]) + ln(EMIT[c][o])
    prev = c
if prev == 'I':
    lp += ln(TRANS[prev]['End'])

print("Log probability of given path:", lp)

n_st = len(ST)
n_obs = len(seq)
V = np.full((n_st, n_obs), -np.inf)
bt = np.zeros((n_st, n_obs), dtype=int)

for i, s in enumerate(ST):
    V[i, 0] = ln(INIT[s]) + ln(EMIT[s][seq[0]])

for k in range(1, n_obs):
    for ci, cs in enumerate(ST):
        max_lp = -math.inf
        best = 0
        for pi, ps in enumerate(ST):
            lp_tmp = V[pi, k-1] + ln(TRANS[ps][cs])
            if lp_tmp > max_lp:
                max_lp = lp_tmp
                best = pi
        V[ci, k] = max_lp + ln(EMIT[cs][seq[k]])
        bt[ci, k] = best

path = []
st = np.argmax(V[:, -1])
path.append(ST[st])
for i in range(n_obs - 1, 0, -1):
    st = bt[st, i]
    path.insert(0, ST[st])

print("\nSequence:           " + seq)
print("Most probable path: " + ''.join(path))


Log probability of given path: -41.21967768602254

Sequence:           CTTCATGTGAAAGCAGACGTAAGTCA
Most probable path: EEEEEEEEEEEEEEEEEEEEEEEEEE
